In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

input_path = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed\train_full_bp_audit_v1.csv")

df = pd.read_csv(input_path)

print(df.shape)
df.head()

(80000, 76)


,patient_id,site_id,triage_nurse_id,arrival_mode,arrival_hour,arrival_day,arrival_month,arrival_season,shift,age,...,pp_recalc,bp_missing_original,pp_category,pp_impossible_flag,pp_extremely_low_flag,pp_very_low_flag,pp_low_flag,pp_wide_flag,pp_extremely_wide_flag,pp_category_label
0,TG-UXRGA9UCO,SITE-TMP-01,NURSE-0033,walk-in,6,Monday,5,spring,morning,43,...,21.5,False,04_pp_20_30_low_plausible,0,0,0,1,0,0,PP 20-30 baja plausible
1,TG-B19DBBS2G,SITE-HEL-01,NURSE-0001,walk-in,6,Thursday,4,spring,morning,72,...,38.3,False,05_pp_30_60_usual,0,0,0,0,0,0,PP 30-60 usual
2,TG-GZ97W7M6V,SITE-HEL-02,NURSE-0005,walk-in,8,Saturday,4,spring,morning,82,...,11.4,False,03_pp_10_20_very_low,0,0,1,1,0,0,PP 10-20 muy baja
3,TG-THIB2TN9Q,SITE-HEL-02,NURSE-0026,police,7,Sunday,3,spring,morning,50,...,82.4,False,06_pp_60_100_wide,0,0,0,0,1,0,PP 60-100 amplia
4,TG-J3U3LQ2QY,SITE-HEL-02,NURSE-0044,walk-in,5,Tuesday,5,spring,night,62,...,64.7,False,06_pp_60_100_wide,0,0,0,0,1,0,PP 60-100 amplia


In [2]:
anthro_cols = ["age", "age_group", "weight_kg", "height_cm", "bmi"]

df[anthro_cols].describe().round(2)

,age,weight_kg,height_cm,bmi
count,80000.00,80000.00,80000.00,80000.00
mean,48.54,74.45,168.62,26.36
std,24.19,21.34,16.59,7.67
min,1.00,2.00,45.00,10.00
25%,29.00,62.00,163.40,21.30
50%,48.00,76.00,171.10,26.00
75%,67.00,88.80,178.10,30.90
max,94.00,148.50,210.00,65.00


In [3]:
df["height_m"] = df["height_cm"] / 100

df["bmi_recalc"] = df["weight_kg"] / (df["height_m"] ** 2)

df["bmi_recalc"] = df["bmi_recalc"].round(2)

df[["weight_kg", "height_cm", "height_m", "bmi", "bmi_recalc"]].head()

,weight_kg,height_cm,height_m,bmi,bmi_recalc
0,52.3,165.4,1.654,19.1,19.12
1,73.3,164.4,1.644,27.1,27.12
2,77.1,183.7,1.837,22.8,22.85
3,49.6,172.6,1.726,16.6,16.65
4,71.9,173.4,1.734,23.9,23.91


In [4]:
df["bmi_diff"] = df["bmi"] - df["bmi_recalc"]
df["bmi_abs_diff"] = df["bmi_diff"].abs()

df[["bmi", "bmi_recalc", "bmi_diff", "bmi_abs_diff"]].describe().round(2)

,bmi,bmi_recalc,bmi_diff,bmi_abs_diff
count,80000.00,80000.00,80000.00,80000.00
mean,26.36,26.35,0.01,0.13
std,7.67,8.19,1.74,1.74
min,10.00,0.52,-168.35,0.00
25%,21.30,21.30,-0.02,0.01
50%,26.00,26.03,0.00,0.03
75%,30.90,30.94,0.03,0.04
max,65.00,233.35,9.48,168.35


In [5]:
conditions = [
    df["bmi_abs_diff"] == 0,
    (df["bmi_abs_diff"] > 0) & (df["bmi_abs_diff"] <= 0.5),
    (df["bmi_abs_diff"] > 0.5) & (df["bmi_abs_diff"] <= 1),
    (df["bmi_abs_diff"] > 1) & (df["bmi_abs_diff"] <= 3),
    df["bmi_abs_diff"] > 3
]

choices = [
    "00_exact_match",
    "01_diff_le_0_5",
    "02_diff_0_5_to_1",
    "03_diff_1_to_3",
    "04_diff_gt_3"
]

df["bmi_diff_category"] = np.select(
    conditions,
    choices,
    default="99_unclassified"
)

df["bmi_discrepancy_flag"] = (df["bmi_abs_diff"] > 1).astype(int)
df["bmi_major_discrepancy_flag"] = (df["bmi_abs_diff"] > 3).astype(int)

In [6]:
resumen_bmi_diff = (
    df["bmi_diff_category"]
    .value_counts(dropna=False)
    .rename_axis("bmi_diff_category")
    .reset_index(name="n")
)

resumen_bmi_diff["%"] = (resumen_bmi_diff["n"] / len(df) * 100).round(2)

resumen_bmi_diff

,bmi_diff_category,n,%
0,01_diff_le_0_5,70731,88.41
1,00_exact_match,8015,10.02
2,04_diff_gt_3,708,0.88
3,03_diff_1_to_3,393,0.49
4,02_diff_0_5_to_1,153,0.19


In [7]:
df["height_implausible_flag"] = (
    (df["height_cm"] < 45) | 
    (df["height_cm"] > 220)
).astype(int)

df["weight_implausible_flag"] = (
    (df["weight_kg"] < 2) | 
    (df["weight_kg"] > 250)
).astype(int)

df["bmi_implausible_flag"] = (
    (df["bmi_recalc"] < 8) | 
    (df["bmi_recalc"] > 80)
).astype(int)

In [8]:
df["is_pediatric"] = (df["age"] < 18).astype(int)
df["is_adult"] = (df["age"] >= 18).astype(int)

In [9]:
df["is_pediatric"] = (df["age"] < 18).astype(int)
df["is_adult"] = (df["age"] >= 18).astype(int)

In [10]:
df[["age", "weight_kg", "height_cm", "bmi"]].describe().round(2)

,age,weight_kg,height_cm,bmi
count,80000.00,80000.00,80000.00,80000.00
mean,48.54,74.45,168.62,26.36
std,24.19,21.34,16.59,7.67
min,1.00,2.00,45.00,10.00
25%,29.00,62.00,163.40,21.30
50%,48.00,76.00,171.10,26.00
75%,67.00,88.80,178.10,30.90
max,94.00,148.50,210.00,65.00


In [11]:
df.nsmallest(20, "weight_kg")[["age", "age_group", "weight_kg", "height_cm", "bmi"]]

,age,age_group,weight_kg,height_cm,bmi
773,3,pediatric,2.0,128.8,10.0
1052,5,pediatric,2.0,154.9,10.0
1117,4,pediatric,2.0,129.5,10.0
2080,9,pediatric,2.0,132.4,10.0
2188,14,pediatric,2.0,106.8,10.0
2683,11,pediatric,2.0,116.3,10.0
3032,14,pediatric,2.0,132.5,10.0
3594,2,pediatric,2.0,107.8,10.0
3632,10,pediatric,2.0,94.0,10.0
4570,8,pediatric,2.0,147.4,10.0


In [12]:
df.nlargest(20, "weight_kg")[["age", "age_group", "weight_kg", "height_cm", "bmi"]]

,age,age_group,weight_kg,height_cm,bmi
75407,44,middle_aged,148.5,155.6,61.3
75612,65,elderly,148.3,159.1,58.6
2770,30,young_adult,146.2,171.6,49.7
44360,45,middle_aged,146.1,171.4,49.8
2613,36,young_adult,144.6,174.3,47.6
48661,24,young_adult,144.1,173.6,47.8
77168,47,middle_aged,144.1,169.6,50.1
28388,16,young_adult,143.7,175.8,46.5
65304,79,elderly,143.6,173.2,47.9
74251,21,young_adult,143.5,180.6,44.0


In [13]:
df.nsmallest(20, "height_cm")[["age", "age_group", "weight_kg", "height_cm", "bmi"]]

,age,age_group,weight_kg,height_cm,bmi
9765,2,pediatric,43.1,45.0,65.0
13339,10,pediatric,42.0,45.0,65.0
70438,2,pediatric,33.7,47.1,65.0
51686,9,pediatric,55.8,48.9,65.0
42399,10,pediatric,42.0,49.1,65.0
6763,8,pediatric,14.8,50.8,57.6
78601,3,pediatric,9.5,51.0,36.6
2991,10,pediatric,46.5,51.2,65.0
13798,5,pediatric,53.2,53.2,65.0
60755,14,pediatric,37.8,54.0,65.0


In [14]:
df.nlargest(20, "height_cm")[["age", "age_group", "weight_kg", "height_cm", "bmi"]]

,age,age_group,weight_kg,height_cm,bmi
4100,84,elderly,69.7,210.0,15.4
5044,58,middle_aged,109.2,210.0,23.7
18793,9,pediatric,67.9,210.0,13.7
22786,9,pediatric,37.1,210.0,10.0
30942,15,pediatric,24.7,210.0,10.0
44062,15,pediatric,13.7,210.0,10.0
45178,2,pediatric,50.1,210.0,10.3
48373,92,elderly,68.1,210.0,14.9
56869,84,elderly,65.6,210.0,14.9
58155,14,pediatric,45.0,210.0,10.1


In [15]:
df.nlargest(20, "bmi")[["age", "age_group", "weight_kg", "height_cm", "bmi"]]

,age,age_group,weight_kg,height_cm,bmi
320,2,pediatric,62.1,69.8,65.0
322,15,pediatric,49.9,87.2,65.0
450,13,pediatric,37.4,73.5,65.0
810,15,pediatric,54.4,85.8,65.0
2282,1,pediatric,59.3,81.6,65.0
2991,10,pediatric,46.5,51.2,65.0
3668,1,pediatric,38.0,61.9,65.0
5110,2,pediatric,57.3,86.5,65.0
5134,11,pediatric,53.7,87.2,65.0
6944,14,pediatric,52.9,75.5,65.0


In [16]:
df["is_pediatric"] = (df["age"] < 18).astype(int)
df["is_adult"] = (df["age"] >= 18).astype(int)

In [17]:
# Recalcular BMI desde peso y talla
df["height_m"] = df["height_cm"] / 100
df["bmi_recalc"] = df["weight_kg"] / (df["height_m"] ** 2)
df["bmi_recalc"] = df["bmi_recalc"].round(2)

# Diferencia entre BMI original y recalculado
df["bmi_diff"] = df["bmi"] - df["bmi_recalc"]
df["bmi_abs_diff"] = df["bmi_diff"].abs()

df[["age", "weight_kg", "height_cm", "bmi", "bmi_recalc", "bmi_abs_diff"]].head()

,age,weight_kg,height_cm,bmi,bmi_recalc,bmi_abs_diff
0,43,52.3,165.4,19.1,19.12,0.02
1,72,73.3,164.4,27.1,27.12,0.02
2,82,77.1,183.7,22.8,22.85,0.05
3,50,49.6,172.6,16.6,16.65,0.05
4,62,71.9,173.4,23.9,23.91,0.01


In [18]:
# Peso pediátrico en piso artificial o imposible
weight_floor_pattern = (
    (df["is_pediatric"] == 1) &
    (df["weight_kg"] <= 2)
)

# BMI piso 10 + peso 2 kg
bmi_floor_weight_floor_pattern = (
    (df["is_pediatric"] == 1) &
    (df["bmi"] <= 10) &
    (df["weight_kg"] <= 2)
)

# BMI techo 65 + talla extremadamente baja
bmi_ceiling_height_floor_pattern = (
    (df["is_pediatric"] == 1) &
    (df["bmi"] >= 65) &
    (df["height_cm"] <= 60)
)

df["anthro_error_likely_flag"] = (
    weight_floor_pattern |
    bmi_floor_weight_floor_pattern |
    bmi_ceiling_height_floor_pattern
).astype(int)

In [19]:
df["anthro_extreme_height_flag"] = (
    (df["is_pediatric"] == 1) &
    (
        (df["height_cm"] <= 60) |
        (df["height_cm"] >= 200)
    )
).astype(int)

In [20]:
df["anthro_bmi_boundary_flag"] = (
    (df["is_pediatric"] == 1) &
    (
        (df["bmi"] <= 10) |
        (df["bmi"] >= 65)
    )
).astype(int)

In [21]:
df["anthro_review_flag"] = (
    (df["anthro_error_likely_flag"] == 1) |
    (df["anthro_extreme_height_flag"] == 1) |
    (df["anthro_bmi_boundary_flag"] == 1)
).astype(int)

In [22]:
conditions = [
    df["anthro_error_likely_flag"].eq(1),
    df["anthro_extreme_height_flag"].eq(1),
    df["anthro_bmi_boundary_flag"].eq(1),
    df["anthro_review_flag"].eq(0)
]

choices = [
    "01_error_likely_combined_pattern",
    "02_extreme_height_review",
    "03_bmi_boundary_review",
    "00_no_obvious_anthro_error"
]

df["anthro_quality_category"] = np.select(
    conditions,
    choices,
    default="99_other_anthro_review"
)

In [23]:
resumen_anthro_flags = pd.DataFrame({
    "flag": [
        "anthro_error_likely_flag",
        "anthro_extreme_height_flag",
        "anthro_bmi_boundary_flag",
        "anthro_review_flag"
    ],
    "n": [
        df["anthro_error_likely_flag"].sum(),
        df["anthro_extreme_height_flag"].sum(),
        df["anthro_bmi_boundary_flag"].sum(),
        df["anthro_review_flag"].sum()
    ]
})

resumen_anthro_flags["%"] = (
    resumen_anthro_flags["n"] / len(df) * 100
).round(2)

resumen_anthro_flags

,flag,n,%
0,anthro_error_likely_flag,108,0.14
1,anthro_extreme_height_flag,36,0.04
2,anthro_bmi_boundary_flag,1147,1.43
3,anthro_review_flag,1158,1.45


In [24]:
resumen_anthro_category = (
    df["anthro_quality_category"]
    .value_counts(dropna=False)
    .rename_axis("anthro_quality_category")
    .reset_index(name="n")
)

resumen_anthro_category["%"] = (
    resumen_anthro_category["n"] / len(df) * 100
).round(2)

resumen_anthro_category

,anthro_quality_category,n,%
0,00_no_obvious_anthro_error,78842,98.55
1,03_bmi_boundary_review,1030,1.29
2,01_error_likely_combined_pattern,108,0.14
3,02_extreme_height_review,20,0.02


In [26]:
cols_revision_anthro = [
    "age",
    "age_group",
    "sex",
    "weight_kg",
    "height_cm",
    "bmi",
    "bmi_recalc",
    "bmi_abs_diff",
    "anthro_error_likely_flag",
    "anthro_extreme_height_flag",
    "anthro_bmi_boundary_flag",
    "anthro_review_flag",
    "anthro_quality_category",
    "triage_acuity",
    "disposition"
]

df.loc[
    df["anthro_review_flag"] == 1,
    cols_revision_anthro
].sort_values(
    ["anthro_quality_category", "age", "weight_kg", "height_cm"]
).head(1000)

,age,age_group,sex,weight_kg,height_cm,bmi,bmi_recalc,bmi_abs_diff,anthro_error_likely_flag,anthro_extreme_height_flag,anthro_bmi_boundary_flag,anthro_review_flag,anthro_quality_category,triage_acuity,disposition
79259,1,pediatric,F,2.0,89.7,10.0,2.49,7.51,1,0,1,1,01_error_likely_combined_pattern,2,discharged
62454,1,pediatric,M,2.0,102.5,10.0,1.90,8.10,1,0,1,1,01_error_likely_combined_pattern,4,discharged
50610,1,pediatric,F,2.0,120.6,10.0,1.38,8.62,1,0,1,1,01_error_likely_combined_pattern,2,admitted
60304,1,pediatric,Other,2.0,139.4,10.0,1.03,8.97,1,0,1,1,01_error_likely_combined_pattern,2,discharged
32771,1,pediatric,M,2.0,196.5,10.0,0.52,9.48,1,0,1,1,01_error_likely_combined_pattern,4,discharged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68367,13,pediatric,M,23.0,153.7,10.0,9.74,0.26,0,0,1,1,03_bmi_boundary_review,2,admitted
20219,13,pediatric,M,24.0,174.6,10.0,7.87,2.13,0,0,1,1,03_bmi_boundary_review,1,admitted
39135,13,pediatric,M,25.1,158.3,10.0,10.02,0.02,0,0,1,1,03_bmi_boundary_review,3,admitted
67579,13,pediatric,M,26.3,174.7,10.0,8.62,1.38,0,0,1,1,03_bmi_boundary_review,2,transferred


In [27]:
df.loc[
    (df["is_pediatric"] == 1) &
    (df["weight_kg"] <= 2),
    cols_revision_anthro
].sort_values("age")

,age,age_group,sex,weight_kg,height_cm,bmi,bmi_recalc,bmi_abs_diff,anthro_error_likely_flag,anthro_extreme_height_flag,anthro_bmi_boundary_flag,anthro_review_flag,anthro_quality_category,triage_acuity,disposition
32771,1,pediatric,M,2.0,196.5,10.0,0.52,9.48,1,0,1,1,01_error_likely_combined_pattern,4,discharged
62454,1,pediatric,M,2.0,102.5,10.0,1.90,8.10,1,0,1,1,01_error_likely_combined_pattern,4,discharged
60304,1,pediatric,Other,2.0,139.4,10.0,1.03,8.97,1,0,1,1,01_error_likely_combined_pattern,2,discharged
50610,1,pediatric,F,2.0,120.6,10.0,1.38,8.62,1,0,1,1,01_error_likely_combined_pattern,2,admitted
79259,1,pediatric,F,2.0,89.7,10.0,2.49,7.51,1,0,1,1,01_error_likely_combined_pattern,2,discharged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36411,15,pediatric,M,2.0,132.9,10.0,1.13,8.87,1,0,1,1,01_error_likely_combined_pattern,5,discharged
28684,15,pediatric,F,2.0,103.3,10.0,1.87,8.13,1,0,1,1,01_error_likely_combined_pattern,3,lama
69340,15,pediatric,F,2.0,168.9,10.0,0.70,9.30,1,0,1,1,01_error_likely_combined_pattern,5,lama
70812,15,pediatric,M,2.0,142.9,10.0,0.98,9.02,1,0,1,1,01_error_likely_combined_pattern,2,admitted


In [28]:
n_weight_2kg = (
    (df["is_pediatric"] == 1) &
    (df["weight_kg"] <= 2)
).sum()

n_weight_2kg

np.int64(92)

In [29]:
df.loc[
    (df["is_pediatric"] == 1) &
    (df["bmi"] <= 10),
    cols_revision_anthro
].sort_values(["age", "weight_kg"]).head(50)

,age,age_group,sex,weight_kg,height_cm,bmi,bmi_recalc,bmi_abs_diff,anthro_error_likely_flag,anthro_extreme_height_flag,anthro_bmi_boundary_flag,anthro_review_flag,anthro_quality_category,triage_acuity,disposition
32771,1,pediatric,M,2.0,196.5,10.0,0.52,9.48,1,0,1,1,01_error_likely_combined_pattern,4,discharged
50610,1,pediatric,F,2.0,120.6,10.0,1.38,8.62,1,0,1,1,01_error_likely_combined_pattern,2,admitted
60304,1,pediatric,Other,2.0,139.4,10.0,1.03,8.97,1,0,1,1,01_error_likely_combined_pattern,2,discharged
62454,1,pediatric,M,2.0,102.5,10.0,1.90,8.10,1,0,1,1,01_error_likely_combined_pattern,4,discharged
79259,1,pediatric,F,2.0,89.7,10.0,2.49,7.51,1,0,1,1,01_error_likely_combined_pattern,2,discharged
73826,1,pediatric,M,2.8,122.1,10.0,1.88,8.12,0,0,1,1,03_bmi_boundary_review,4,discharged
55442,1,pediatric,F,3.3,129.2,10.0,1.98,8.02,0,0,1,1,03_bmi_boundary_review,2,observation
28679,1,pediatric,F,5.7,149.6,10.0,2.55,7.45,0,0,1,1,03_bmi_boundary_review,2,observation
58664,1,pediatric,F,5.8,153.3,10.0,2.47,7.53,0,0,1,1,03_bmi_boundary_review,1,admitted
44740,1,pediatric,M,6.5,102.4,10.0,6.20,3.80,0,0,1,1,03_bmi_boundary_review,3,admitted


In [30]:
df.loc[
    (df["is_pediatric"] == 1) &
    (df["bmi"] >= 65),
    cols_revision_anthro
].sort_values(["age", "height_cm"]).head(50)

,age,age_group,sex,weight_kg,height_cm,bmi,bmi_recalc,bmi_abs_diff,anthro_error_likely_flag,anthro_extreme_height_flag,anthro_bmi_boundary_flag,anthro_review_flag,anthro_quality_category,triage_acuity,disposition
18249,1,pediatric,F,45.4,56.9,65.0,140.23,75.23,1,1,1,1,01_error_likely_combined_pattern,4,discharged
34366,1,pediatric,F,41.2,58.2,65.0,121.63,56.63,1,1,1,1,01_error_likely_combined_pattern,2,admitted
3668,1,pediatric,M,38.0,61.9,65.0,99.18,34.18,0,0,1,1,03_bmi_boundary_review,4,discharged
37720,1,pediatric,F,46.8,66.5,65.0,105.83,40.83,0,0,1,1,03_bmi_boundary_review,3,lama
9239,1,pediatric,M,41.1,68.0,65.0,88.88,23.88,0,0,1,1,03_bmi_boundary_review,5,discharged
25333,1,pediatric,F,42.5,69.2,65.0,88.75,23.75,0,0,1,1,03_bmi_boundary_review,3,admitted
64660,1,pediatric,F,40.1,70.3,65.0,81.14,16.14,0,0,1,1,03_bmi_boundary_review,3,admitted
49721,1,pediatric,F,38.9,71.3,65.0,76.52,11.52,0,0,1,1,03_bmi_boundary_review,4,admitted
10195,1,pediatric,M,39.5,74.0,65.0,72.13,7.13,0,0,1,1,03_bmi_boundary_review,3,observation
2282,1,pediatric,F,59.3,81.6,65.0,89.06,24.06,0,0,1,1,03_bmi_boundary_review,4,discharged


In [31]:
df.loc[
    (df["is_pediatric"] == 1) &
    (
        (df["height_cm"] <= 60) |
        (df["height_cm"] >= 200)
    ),
    cols_revision_anthro
].sort_values(["age", "height_cm"]).head(100)

,age,age_group,sex,weight_kg,height_cm,bmi,bmi_recalc,bmi_abs_diff,anthro_error_likely_flag,anthro_extreme_height_flag,anthro_bmi_boundary_flag,anthro_review_flag,anthro_quality_category,triage_acuity,disposition
18249,1,pediatric,F,45.4,56.9,65.0,140.23,75.23,1,1,1,1,01_error_likely_combined_pattern,4,discharged
34366,1,pediatric,F,41.2,58.2,65.0,121.63,56.63,1,1,1,1,01_error_likely_combined_pattern,2,admitted
5763,1,pediatric,M,10.4,207.9,10.0,2.41,7.59,0,1,1,1,02_extreme_height_review,4,discharged
8364,1,pediatric,F,25.0,209.6,10.0,5.69,4.31,0,1,1,1,02_extreme_height_review,3,observation
9765,2,pediatric,F,43.1,45.0,65.0,212.84,147.84,1,1,1,1,01_error_likely_combined_pattern,4,discharged
70438,2,pediatric,M,33.7,47.1,65.0,151.91,86.91,1,1,1,1,01_error_likely_combined_pattern,1,admitted
45178,2,pediatric,M,50.1,210.0,10.3,11.36,1.06,0,1,0,1,02_extreme_height_review,2,lama
78601,3,pediatric,M,9.5,51.0,36.6,36.52,0.08,0,1,0,1,02_extreme_height_review,2,admitted
22570,3,pediatric,F,33.8,208.8,10.0,7.75,2.25,0,1,1,1,02_extreme_height_review,5,discharged
43780,4,pediatric,M,55.5,58.5,65.0,162.17,97.17,1,1,1,1,01_error_likely_combined_pattern,3,discharged


In [32]:
pd.crosstab(
    df["anthro_quality_category"],
    df["triage_acuity"],
    normalize="index"
).round(3) * 100

triage_acuity,1,2,3,4,5
anthro_quality_category,,,,,
00_no_obvious_anthro_error,4.0,16.8,36.2,28.8,14.3
01_error_likely_combined_pattern,3.7,25.9,36.1,18.5,15.7
02_extreme_height_review,5.0,25.0,30.0,25.0,15.0
03_bmi_boundary_review,4.5,19.2,34.9,28.7,12.7


In [33]:
pd.crosstab(
    df["anthro_quality_category"],
    df["triage_acuity"],
    normalize="index"
).round(3) * 100

triage_acuity,1,2,3,4,5
anthro_quality_category,,,,,
00_no_obvious_anthro_error,4.0,16.8,36.2,28.8,14.3
01_error_likely_combined_pattern,3.7,25.9,36.1,18.5,15.7
02_extreme_height_review,5.0,25.0,30.0,25.0,15.0
03_bmi_boundary_review,4.5,19.2,34.9,28.7,12.7


In [34]:
pd.crosstab(
    df["anthro_quality_category"],
    df["disposition"],
    normalize="index"
).round(3) * 100

disposition,admitted,deceased,discharged,lama,lwbs,observation,transferred
anthro_quality_category,,,,,,,
00_no_obvious_anthro_error,30.7,0.5,48.8,3.4,4.6,5.4,6.5
01_error_likely_combined_pattern,30.6,0.0,39.8,9.3,4.6,8.3,7.4
02_extreme_height_review,30.0,5.0,40.0,5.0,5.0,5.0,10.0
03_bmi_boundary_review,32.7,0.3,46.1,3.8,4.0,6.2,6.9


In [35]:
from pathlib import Path

output_dir = Path(r"C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "train_full_anthropometry_audit_v1.csv"

df.to_csv(output_path, index=False)

print(f"Archivo guardado en: {output_path}")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

Archivo guardado en: C:\Users\pablo\OneDrive\Desktop\triagegeist\data\processed\parcial_processed\train_full_anthropometry_audit_v1.csv
Filas: 80000
Columnas: 93


In [36]:
df_check = pd.read_csv(output_path)

print(df_check.shape)

df_check[[
    "age",
    "weight_kg",
    "height_cm",
    "bmi",
    "bmi_recalc",
    "anthro_error_likely_flag",
    "anthro_extreme_height_flag",
    "anthro_bmi_boundary_flag",
    "anthro_review_flag",
    "anthro_quality_category"
]].head()

(80000, 93)


,age,weight_kg,height_cm,bmi,bmi_recalc,anthro_error_likely_flag,anthro_extreme_height_flag,anthro_bmi_boundary_flag,anthro_review_flag,anthro_quality_category
0,43,52.3,165.4,19.1,19.12,0,0,0,0,00_no_obvious_anthro_error
1,72,73.3,164.4,27.1,27.12,0,0,0,0,00_no_obvious_anthro_error
2,82,77.1,183.7,22.8,22.85,0,0,0,0,00_no_obvious_anthro_error
3,50,49.6,172.6,16.6,16.65,0,0,0,0,00_no_obvious_anthro_error
4,62,71.9,173.4,23.9,23.91,0,0,0,0,00_no_obvious_anthro_error


In [37]:
adultos_peso_10_o_menos = df.loc[
    (df["is_adult"] == 1) &
    (df["weight_kg"] <= 10),
    [
        "age",
        "age_group",
        "sex",
        "weight_kg",
        "height_cm",
        "bmi",
        "bmi_recalc",
        "triage_acuity",
        "disposition"
    ]
].sort_values(["age", "weight_kg"])

print(f"Cantidad de adultos con peso <= 10 kg: {len(adultos_peso_10_o_menos)}")

adultos_peso_10_o_menos

Cantidad de adultos con peso <= 10 kg: 6


,age,age_group,sex,weight_kg,height_cm,bmi,bmi_recalc,triage_acuity,disposition
59665,20,young_adult,M,3.5,166.2,10.0,1.27,4,discharged
61938,26,young_adult,M,4.9,171.9,10.0,1.66,2,discharged
37826,49,middle_aged,F,9.7,176.8,10.0,3.10,2,lwbs
30630,56,middle_aged,F,9.4,166.6,10.0,3.39,2,transferred
66049,84,elderly,F,2.8,171.8,10.0,0.95,3,discharged
60153,88,elderly,F,8.7,181.3,10.0,2.65,4,discharged


In [38]:
adultos_talla_60_o_menos = df.loc[
    (df["is_adult"] == 1) &
    (df["height_cm"] <= 60),
    [
        "age",
        "age_group",
        "sex",
        "weight_kg",
        "height_cm",
        "bmi",
        "bmi_recalc",
        "triage_acuity",
        "disposition"
    ]
].sort_values(["age", "height_cm"])

print(f"Cantidad de adultos con talla <= 60 cm: {len(adultos_talla_60_o_menos)}")

adultos_talla_60_o_menos

Cantidad de adultos con talla <= 60 cm: 0


,age,age_group,sex,weight_kg,height_cm,bmi,bmi_recalc,triage_acuity,disposition


# Conclusiones auditoría antropométrica

## Objetivo

Evaluar la coherencia interna de las variables antropométricas antes de la etapa de imputación, limpieza final y modelado.

Las variables revisadas fueron:

- `weight_kg`
- `height_cm`
- `bmi`

Además, se recalculó el IMC mediante peso y talla para compararlo con el IMC original del dataset.

## Hallazgos principales

### 1. El BMI fue tratado como variable derivada

Se creó `bmi_recalc` a partir de `weight_kg` y `height_cm`.

Esto permite comparar el IMC original con el IMC matemáticamente esperado y detectar posibles inconsistencias internas.

### 2. Se identificaron patrones antropométricos extremos

Durante la revisión exploratoria se observaron valores compatibles con bordes artificiales o datos no confiables, especialmente en población pediátrica.

Los patrones principales fueron:

- Peso pediátrico extremadamente bajo, especialmente `weight_kg <= 2`.
- Tallas pediátricas extremadamente bajas.
- Tallas pediátricas extremadamente altas.
- Valores de BMI en bordes llamativos, especialmente cercanos a 10 y 65.

### 3. No se corrigieron manualmente los valores extremos

Dado que algunos valores extremos podrían representar artefactos del dataset o señales generadas de forma sintética, no se eliminaron ni corrigieron automáticamente.

En esta etapa solo se crearon flags para marcar registros que requieren cautela.

### 4. Se crearon flags antropométricas simples

Se conservaron pocas flags para evitar sobrecargar el dataset:

- `anthro_error_likely_flag`
- `anthro_extreme_height_flag`
- `anthro_bmi_boundary_flag`
- `anthro_review_flag`
- `anthro_quality_category`

Estas variables permiten diferenciar registros sin errores evidentes de aquellos con patrones antropométricos extremos o probablemente artificiales.

## Decisión metodológica

Para etapas posteriores:

- Se conservarán `weight_kg` y `height_cm`.
- Se conservará `bmi_recalc` como alternativa al BMI original.
- El BMI original podrá compararse contra `bmi_recalc`.
- Las flags antropométricas se mantendrán como señales de calidad de datos.
- No se realizará corrección manual caso a caso en esta etapa.

## Conclusión

La auditoría antropométrica permitió identificar valores extremos y posibles bordes artificiales del dataset, sin transformar esta etapa en una limpieza manual exhaustiva.

La decisión final será conservar la información original, agregar variables derivadas y flags de calidad, y evaluar posteriormente si estas variables aportan al modelo predictivo.